# Whole-well brightfield Z-stack → OME-Zarr / OME-TIFF
Tile-scan the whole well (22×22) and acquire a 5-plane Z-stack at each tile, assembled **in memory** into a `(Z, Y, X)` mosaic stack — no per-tile files on disk. Saved directly to OME-Zarr / OME-TIFF with the physical voxel size.

## 1. Connect

In [ ]:
import sys, importlib
import numpy as np
import matplotlib.pyplot as plt
import tifffile, re, time
from pathlib import Path
from pycromanager import Core, Studio

sys.path.insert(0, r'C:\Users\aifadmin\scapecontrol\repo')
import opm_acquisition; importlib.reload(opm_acquisition)

core = Core()
studio = Studio()
print('Connected â€” camera:', core.get_camera_device())

## 2. Switch to widefield and preview
Sets widefield/LED mode, `LED_INTENSITY_PCT` and `EXPOSURE_MS` (reused by the acquisition). Run before acquiring.

In [ ]:
from opm_acquisition import switch_to_widefield

LED_INTENSITY_PCT = 20
EXPOSURE_MS       = 20.0

switch_to_widefield(core)
core.set_property('LED:L:37:4', 'LED Intensity(%)', str(LED_INTENSITY_PCT))
core.set_exposure(EXPOSURE_MS)

core.snap_image()
tagged = core.get_tagged_image()
img = np.reshape(tagged.pix, [tagged.tags['Height'], tagged.tags['Width']]).astype(np.float32)

print(f'Camera: {core.get_camera_device()}  |  Shutter: {core.get_shutter_device()}')
print(f'Shape: {img.shape}  |  Min: {img.min():.0f}  Max: {img.max():.0f}  Mean: {img.mean():.0f}')

p1, p99 = np.percentile(img, (1, 99))
img_display = np.clip((img - p1) / (p99 - p1 + 1e-9), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_display, cmap='gray')
axes[0].set_title(f'Preview â€” LED {LED_INTENSITY_PCT}%  exp {EXPOSURE_MS} ms')
axes[0].axis('off')
axes[1].hist(img.ravel(), bins=256, color='gray')
axes[1].set_title('Pixel intensity histogram')
axes[1].set_xlabel('Intensity (raw)')
plt.tight_layout()
plt.show()

## 3. Settings — grid + Z-stack + output

In [ ]:
# --- Output (the OME file is written here; no intermediate per-tile files) ---
OUTPUT_DIR = Path(r'Y:\Collaborations\aif-botton-collaboration\Johannes\20260817\ome zarr test\z_stack')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NAME       = 'wellstack'

# --- Camera (widefield PCO) ---
PIXEL_SIZE_UM = 0.36
CAMERA_PX     = 1024
FOV_UM        = CAMERA_PX * PIXEL_SIZE_UM          # ~369 um

# --- XY grid (whole well) ---
RANGE_X_UM = 6900     # -> 22 cols
RANGE_Y_UM = 6900     # -> 22 rows
OVERLAP    = 0.10
step_um    = FOV_UM * (1 - OVERLAP)
CUSTOM_GRID = dict(x_start=-RANGE_X_UM/2, x_end=RANGE_X_UM/2,
                   y_start=-RANGE_Y_UM/2, y_end=RANGE_Y_UM/2, step_um=step_um)

# --- Z-stack at each tile ---
Z_DEVICE   = core.get_focus_device()               # 'ZStage:Z:32'
N_PLANES   = 5
Z_STEP_UM  = 10.0
Z_SETTLE_S = 0.05
offsets    = (np.arange(N_PLANES) - (N_PLANES - 1) / 2) * Z_STEP_UM   # -20..+20

n_cols = round(RANGE_X_UM / step_um) + 1
n_rows = round(RANGE_Y_UM / step_um) + 1
mp_gb  = n_cols * n_rows * CAMERA_PX * CAMERA_PX * 2 * N_PLANES / 1e9
print(f'Grid     : {n_cols} x {n_rows} = {n_cols*n_rows} tiles   (step {step_um:.0f} um, {OVERLAP*100:.0f}% overlap)')
print(f'Z-stack  : {N_PLANES} planes @ {Z_STEP_UM} um  ->  offsets {offsets.tolist()} um')
print(f'Snaps    : {n_cols*n_rows*N_PLANES}   mosaic stack ~ {mp_gb:.1f} GB in RAM')
print(f'Output   : {OUTPUT_DIR}')

## 4. Preview grid

In [ ]:
importlib.reload(opm_acquisition)
from opm_acquisition import _snake_grid

x_cur, y_cur = core.get_x_position(), core.get_y_position()
events = _snake_grid(CUSTOM_GRID, center_x=x_cur, center_y=y_cur)
xs = [e['x'] for e in events]; ys = [e['y'] for e in events]

plt.figure(figsize=(7, 7))
plt.plot(xs, ys, 'o-', ms=3)
plt.plot(x_cur, y_cur, 'r*', ms=14, label='center (current pos)')
plt.xlabel('X (um)'); plt.ylabel('Y (um)')
plt.title(f'{len(events)} tiles  step={step_um:.0f} um'); plt.legend()
plt.axis('equal'); plt.grid(True); plt.show()

## 5. Acquire whole-well Z-stack (in memory)
Outer loop = XY tiles (snake), inner loop = 5 Z-planes. Each snap is placed straight into the mosaic stack; nothing is written to disk here. **Run step 2 first** (widefield mode + exposure). Park the stage at the well centre and focus the middle plane before starting.

In [ ]:
from opm_acquisition import _snake_grid

XY_STAGE = core.get_xy_stage_device()
core.set_property('LED:L:37:4', 'LED Intensity(%)', str(LED_INTENSITY_PCT))
core.set_exposure(EXPOSURE_MS)

x_cur, y_cur = core.get_x_position(), core.get_y_position()
z_center     = core.get_position(Z_DEVICE)
events       = _snake_grid(CUSTOM_GRID, center_x=x_cur, center_y=y_cur)
rows         = sorted({e['axes']['row'] for e in events})
cols         = sorted({e['axes']['col'] for e in events})
max_row      = max(rows)
z_positions  = z_center + offsets

stack = None
t0 = time.time()
for k, ev in enumerate(events):
    core.set_xy_position(ev['x'], ev['y'])
    core.wait_for_device(XY_STAGE)
    time.sleep(0.05)
    r, c = ev['axes']['row'], ev['axes']['col']

    for zi, z in enumerate(z_positions):
        core.set_position(Z_DEVICE, float(z))
        core.wait_for_device(Z_DEVICE)
        time.sleep(Z_SETTLE_S)
        core.snap_image()
        tg  = core.get_tagged_image()
        img = np.reshape(tg.pix, [tg.tags['Height'], tg.tags['Width']])
        if stack is None:
            h, w  = img.shape
            stack = np.zeros((N_PLANES, len(rows) * h, len(cols) * w), dtype=img.dtype)
        mr = (max_row - r) * h    # higher Y row -> top
        mc = c * w                # col 0 -> left
        stack[zi, mr:mr + h, mc:mc + w] = img

    if c == 0:
        print(f'  row {r+1}/{len(rows)}   {k*N_PLANES}/{len(events)*N_PLANES} snaps   {time.time()-t0:.0f}s')

core.set_xy_position(x_cur, y_cur); core.wait_for_device(XY_STAGE)
core.set_position(Z_DEVICE, z_center); core.wait_for_device(Z_DEVICE)
print(f'Done: {stack.shape} {stack.dtype} in {time.time()-t0:.0f}s (in memory)')

# montage preview (downsampled)
fig, axes = plt.subplots(1, N_PLANES, figsize=(3 * N_PLANES, 3))
for i, ax in enumerate(np.atleast_1d(axes)):
    sm = stack[i, ::32, ::32].astype(np.float32)
    p1, p99 = np.percentile(sm, (1, 99))
    ax.imshow(np.clip((sm - p1) / (p99 - p1 + 1e-9), 0, 1), cmap='gray')
    ax.set_title(f'z={offsets[i]:+.0f} um'); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Save as OME-Zarr
Uses an `OMEZarrImage` / `OMEZarrMultiscale` API (the same call style as ome-zarr 0.18) implemented on top of the installed ome-zarr 0.16 `write_multiscale`. Run the shim cell once, then the save cell.

In [ ]:
# --- OME-Zarr writer: OMEZarrImage / OMEZarrMultiscale ---
# Same call style as ome-zarr 0.18, implemented on the installed ome-zarr 0.16
# write_multiscale. If you later upgrade to >=0.18 you can delete this cell and
# use `from ome_zarr import OMEZarrImage, OMEZarrMultiscale` instead.
import shutil
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_multiscale

_UNIT = {'micrometer': 'micrometer', 'um': 'micrometer', '\u00b5m': 'micrometer',
         'nanometer': 'nanometer', 'nm': 'nanometer',
         'millimeter': 'millimeter', 'mm': 'millimeter'}
_TYPE = {'x': 'space', 'y': 'space', 'z': 'space', 'c': 'channel', 't': 'time'}


class OMEZarrImage:
    """Labelled array: data + axis names + per-axis physical scale (+ units)."""
    def __init__(self, data, axes, scale, axes_units=None):
        self.data = np.asarray(data)
        self.axes = list(axes)
        self.scale = dict(scale)
        self.axes_units = dict(axes_units or {})
        if self.data.ndim != len(self.axes):
            raise ValueError(f'axes {self.axes} do not match data.ndim={self.data.ndim}')


class OMEZarrMultiscale:
    """Build a multiscale pyramid and write it as OME-Zarr.

    Downsampling is applied to the x/y axes only (z/c/t kept), which is what a
    thin z-stack overview wants. `scale_factors` are absolute factors from full
    resolution: (2, 4, 8) -> levels [full, /2, /4, /8].
    method='nearest' strides (fast); 'resize' uses skimage (slower, upcasts).
    """
    def __init__(self, image, scale_factors=(2, 4, 8), method='nearest',
                 channel_names=None, channel_colors=None, contrast_limits=None,
                 chunks=None):
        self.image = image
        self.scale_factors = tuple(scale_factors)
        self.method = method
        self.channel_names = channel_names
        self.channel_colors = channel_colors
        self.contrast_limits = contrast_limits
        self.chunks = chunks

    def _downsample(self, data, f):
        ax = self.image.axes
        if self.method == 'resize':
            from skimage.transform import resize
            shp = [(s // f if ax[i] in ('x', 'y') else s) for i, s in enumerate(data.shape)]
            return resize(data, shp, order=1, preserve_range=True,
                          anti_aliasing=False).astype(data.dtype)
        sl = tuple(slice(None, None, f) if ax[i] in ('x', 'y') else slice(None)
                   for i in range(data.ndim))
        return data[sl]

    def _pyramid(self):
        d = self.image.data
        return [d] + [self._downsample(d, f) for f in self.scale_factors]

    def _axes_meta(self):
        out = []
        for name in self.image.axes:
            a = {'name': name, 'type': _TYPE.get(name, 'space')}
            u = self.image.axes_units.get(name)
            if u:
                a['unit'] = _UNIT.get(u, u)
            out.append(a)
        return out

    def _transforms(self, pyr):
        ax, full = self.image.axes, pyr[0].shape
        return [[{'type': 'scale',
                  'scale': [self.image.scale[ax[i]] * (full[i] / lv.shape[i])
                            for i in range(len(ax))]}] for lv in pyr]

    def _omero(self, name):
        ax = self.image.axes
        n = self.image.data.shape[ax.index('c')] if 'c' in ax else 1
        names = self.channel_names or [f'ch{i}' for i in range(n)]
        colors = self.channel_colors or ['FFFFFF'] * n
        chans = []
        for i in range(n):
            ch = {'label': names[i], 'color': colors[i % len(colors)], 'active': True}
            if self.contrast_limits:
                lo, hi = self.contrast_limits[i]
                ch['window'] = {'start': lo, 'end': hi, 'min': lo, 'max': hi}
            chans.append(ch)
        return {'name': name, 'channels': chans}

    def to_ome_zarr(self, path, overwrite=True):
        p = Path(path)
        if overwrite and p.exists():
            shutil.rmtree(p)
        pyr = self._pyramid()
        store = parse_url(str(p), mode='w').store
        root = zarr.group(store=store)
        so = dict(chunks=self.chunks) if self.chunks else None
        write_multiscale(pyramid=pyr, group=root, axes=self._axes_meta(),
                         coordinate_transformations=self._transforms(pyr),
                         storage_options=so)
        root.attrs['omero'] = self._omero(p.stem)
        return p

print('OMEZarrImage / OMEZarrMultiscale ready')

In [ ]:
# Save the whole-well Z-stack as OME-Zarr
image = OMEZarrImage(
    data=stack,
    axes=['z', 'y', 'x'],
    scale={'z': Z_STEP_UM, 'y': PIXEL_SIZE_UM, 'x': PIXEL_SIZE_UM},
    axes_units={'z': 'micrometer', 'y': 'micrometer', 'x': 'micrometer'},
)

multiscales = OMEZarrMultiscale(
    image=image,
    scale_factors=(2, 4, 8),
    method='nearest',                    # 'resize' also supported (slower on big mosaics)
    channel_names=['brightfield'],       # optional
    channel_colors=['FFFFFF'],           # optional
    # contrast_limits=[(0, 4096)],       # optional
    chunks=(1, 1024, 1024),
)

ZARR_PATH = OUTPUT_DIR / f'{NAME}.ome.zarr'
multiscales.to_ome_zarr(ZARR_PATH)

print(f'Saved OME-Zarr : {ZARR_PATH}')
print(f'  data {stack.shape} {stack.dtype}   voxel z={Z_STEP_UM} xy={PIXEL_SIZE_UM} um')
print(f'  levels : {[lv.shape for lv in multiscales._pyramid()]}')

## 7. Save as OME-TIFF (alternative)
Single BigTIFF (~5 GB for the whole well). OME-Zarr above is usually the better choice at this size; use this if a tool needs TIFF.

In [ ]:
TIFF_PATH = OUTPUT_DIR / f'{NAME}.ome.tif'
tifffile.imwrite(
    str(TIFF_PATH), stack, photometric='minisblack', bigtiff=True,
    metadata={'axes': 'ZYX',
              'PhysicalSizeZ': Z_STEP_UM, 'PhysicalSizeZUnit': 'um',
              'PhysicalSizeX': PIXEL_SIZE_UM, 'PhysicalSizeY': PIXEL_SIZE_UM,
              'PhysicalSizeXUnit': 'um', 'PhysicalSizeYUnit': 'um'},
)
print(f'Saved OME-TIFF : {TIFF_PATH}   {stack.shape} {stack.dtype}')